# Comprehensive Gen Python Bindings Demo

This notebook exhaustively demonstrates all functionality available in the Gen Python bindings.
It covers: initialization, imports, exports, updates, search, indexing, and interactive visualization.

---

## 1. Setup and Repository Initialization

In [1]:
import gen
import tempfile
import pathlib

REPO_ROOT = pathlib.Path(gen.__file__).parents[3]
FIXTURES = REPO_ROOT / "fixtures"

WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-api-demo-"))
print(f"Working directory: {WORK_DIR}")

repo = gen.Repository(str(WORK_DIR))
print(f"Repository initialized: {repo}")
print(f"  gen_dir: {repo.gen_dir}")
print(f"  db_path: {repo.db_path}")

Working directory: /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-api-demo-blgeh_0t
Repository initialized: <builtins.Repository object at 0x10c00a1a0>
  gen_dir: /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-api-demo-blgeh_0t/.gen
  db_path: /private/var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-api-demo-blgeh_0t/.gen/default.db


### 1.2 Using `gen.get_gen_dir()` to find existing repository

In [2]:
import os
original_cwd = os.getcwd()
os.chdir(WORK_DIR)
gen_dir = gen.get_gen_dir()
print(f"Found .gen directory: {gen_dir}")
os.chdir(original_cwd)

Found .gen directory: /private/var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/gen-api-demo-blgeh_0t/.gen


## 2. Import Functions

Gen supports importing from multiple file formats: FASTA, GFA, GenBank, and Library.

### 2.1 Import FASTA

In [3]:
SIMPLE_FA = FIXTURES / "simple.fa"
print(f"FASTA file: {SIMPLE_FA}")
print(SIMPLE_FA.read_text())

FASTA file: /Users/bvh/git/gen/fixtures/simple.fa
>m123
ATCGATCGATCGATCGATCGGGAACACACAGAGA



In [4]:
repo.import_fasta

<function Repository.import_fasta(filename, name=None, sample=None, shallow=False)>

In [5]:
repo.import_fasta(str(SIMPLE_FA), name="simple", sample="demo", shallow=False)
block_groups = repo.get_block_groups()
print(f"Imported {len(block_groups)} block group(s)")
for bg in block_groups:
    print(f"  {bg}")

Imported 1 block group(s)
  BlockGroup(5770f9ea63f9098d0c41e054d33f1e631318cdce4a89f103dd5122db92a13ace, simple, demo, m123)


### 2.2 Import GFA (Graph Fragment Assembly)

In [6]:
SIMPLE_GFA = FIXTURES / "simple.gfa"
print(f"GFA file: {SIMPLE_GFA}")
print(SIMPLE_GFA.read_text())

GFA file: /Users/bvh/git/gen/fixtures/simple.gfa
H	VN:Z:1.2
S	1	ATC	SN:Z:m123	SO:i:0	SR:i:0
S	2	GATCGATCGA	SN:Z:m123	SO:i:3	SR:i:0
S	3	TCGATCGGG	SN:Z:m123	SO:i:13	SR:i:0
S	4	AACACACAGAGA	SN:Z:m123	SO:i:22	SR:i:0
L	1	+	2	+	*
L	2	+	3	+	*
L	3	+	4	+	*
P	m123	1+,2+,3+,4+	4M



In [7]:
repo.import_gfa(str(SIMPLE_GFA), name="graph_example", sample="demo")
block_groups = repo.get_block_groups()
print(f"Total block groups: {len(block_groups)}")

Total block groups: 2


### 2.3 Import GenBank

In [8]:
INSERTION_GB = FIXTURES / "geneious_genbank" / "insertion.gb"
print(f"GenBank file: {INSERTION_GB}")
print(INSERTION_GB.read_text()[:500] + "...")

GenBank file: /Users/bvh/git/gen/fixtures/geneious_genbank/insertion.gb
LOCUS       insertion               8302 bp    DNA     circular     27-NOV-2024
DEFINITION  Cloning vector pBeloBAC11, complete sequence.
ACCESSION   urn.local...12f-ii1u066
KEYWORDS    .
SOURCE      Cloning vector pBeloBAC11
  ORGANISM  Cloning vector pBeloBAC11
            other sequences; artificial sequences; vectors.
REFERENCE   1  (bases 1 to 7507)
  AUTHORS   New England Biolabs.
  TITLE     Direct Submission
  JOURNAL   Submitted (19-OCT-2007) Research Department, New England Biolabs,
  ...


In [9]:
repo.import_genbank(str(INSERTION_GB), name="insertion_example", collection="demo")
block_groups = repo.get_block_groups()
print(f"Total block groups: {len(block_groups)}")

TypeError: Repository.import_genbank() got an unexpected keyword argument 'collection'

### 2.4 Import Library (combinatorial design)

In [ ]:
COMBO_CSV = FIXTURES / "combinatorial_design.csv"
print(f"Library file: {COMBO_CSV}")
print(COMBO_CSV.read_text())

In [ ]:
repo.import_library(str(COMBO_CSV), name="combinatorial_lib", collection="demo")
block_groups = repo.get_block_groups()
print(f"Total block groups: {len(block_groups)}")
for bg in block_groups:
    print(f"  {bg}")

## 3. Export Functions

Gen supports exporting to FASTA, GFA, and GenBank formats.

In [ ]:
block_group = repo.get_block_groups()[0]
print(f"Exporting: {block_group}")

### 3.1 Export FASTA

In [ ]:
EXPORTED_FA = WORK_DIR / "exported.fa"
gen.export_fasta(repo, block_group, str(EXPORTED_FA))
print(f"Exported FASTA:")
print(EXPORTED_FA.read_text())

### 3.2 Export GFA

In [ ]:
EXPORTED_GFA = WORK_DIR / "exported.gfa"
gen.export_gfa(repo, block_group, str(EXPORTED_GFA))
print(f"Exported GFA:")
print(EXPORTED_GFA.read_text())

### 3.3 Export GenBank

In [ ]:
EXPORTED_GB = WORK_DIR / "exported.gb"
gen.export_genbank(repo, block_group, str(EXPORTED_GB))
print(f"Exported GenBank:")
print(EXPORTED_GB.read_text())

## 4. Update Functions

Updates modify existing block groups with new sequences, variants, or graph changes.

### 4.1 Update with Sequence (direct insertion)

In [ ]:
bg_for_update = repo.get_block_groups()[0]
print(f"Updating: {bg_for_update}")

gen.update_with_sequence(
    repo,
    bg_for_update,
    "ATCGATCGATCGATCGATCGATCGATCGATCGATCG",
    "inserted_sequence",
    10,
    "INS"
)
print("Updated block group with sequence")

### 4.2 Update with FASTA

In [ ]:
PARTS_FA = FIXTURES / "parts.fa"
print(f"Parts FASTA: {PARTS_FA}")
print(PARTS_FA.read_text())

In [ ]:
PARTS_BG = repo.get_block_groups()[0]
gen.update_with_fasta(repo, PARTS_BG, str(PARTS_FA), "add_parts")
print(f"Updated with parts")

### 4.3 Update with VCF (variant calling)

In [ ]:
SIMPLE_VCF = FIXTURES / "simple.vcf"
print(f"VCF file: {SIMPLE_VCF}")
print(SIMPLE_VCF.read_text())

In [ ]:
bg_for_vcf = repo.get_block_groups()[0]
gen.update_with_vcf(repo, bg_for_vcf, str(SIMPLE_VCF), "apply_variants")
print(f"Updated with VCF")

### 4.4 Update with GFA

In [ ]:
WALK_GFA = FIXTURES / "walk.gfa"
print(f"Walk GFA: {WALK_GFA}")
print(WALK_GFA.read_text())

In [ ]:
bg_for_gfa = repo.get_block_groups()[0]
gen.update_with_gfa(repo, bg_for_gfa, str(WALK_GFA), "merge_walk")
print(f"Updated with GFA")

### 4.5 Update with GenBank

In [ ]:
DELETION_GB = FIXTURES / "geneious_genbank" / "deletion.gb"
bg_for_gb = repo.get_block_groups()[0]
gen.update_with_genbank(repo, bg_for_gb, str(DELETION_GB), "apply_deletion")
print(f"Updated with GenBank")

### 4.6 Update with GAF (graph alignment format)

In [ ]:
CHR22_GAF = FIXTURES / "chr22_het.gaf"
print(f"GAF file: {CHR22_GAF}")
print(CHR22_GAF.read_text()[:500])

In [ ]:
bg_for_gaf = repo.get_block_groups()[0]
gen.update_with_gaf(repo, bg_for_gaf, str(CHR22_GAF), "align_gaf")
print(f"Updated with GAF")

### 4.7 Update with Library

In [ ]:
SINGLE_COL_CSV = FIXTURES / "single_column_design.csv"
print(f"Library CSV: {SINGLE_COL_CSV}")
print(SINGLE_COL_CSV.read_text())

In [ ]:
bg_for_lib = repo.get_block_groups()[0]
gen.update_with_library(repo, bg_for_lib, str(SINGLE_COL_CSV), "apply_library")
print(f"Updated with library")

## 5. Search and Indexing

Gen provides fast sequence search with optional indexing.

### 5.1 Search without Index (full scan)

In [ ]:
test_bg = repo.get_block_groups()[0]
print(f"Searching in: {test_bg}")

query = "ATCG"
matches = repo.search(test_bg, query)
print(f"Found {len(matches)} match(es) for '{query}'")
for m in matches[:3]:
    print(f"  {m}")

### 5.2 Build Search Index

In [ ]:
repo.build_index(k=8)
print("Search index built successfully")

index_dir = pathlib.Path(repo.gen_dir) / "search_index"
index_files = list(index_dir.glob("*.bin")) if index_dir.exists() else []
print(f"Index files created: {len(index_files)}")

### 5.3 Search with Index

In [ ]:
matches = repo.search(test_bg, query)
print(f"Found {len(matches)} match(es) for '{query}' (with index)")
for m in matches[:3]:
    print(f"  {m}")

### 5.4 Block-group level index operations

In [ ]:
test_bg = repo.get_block_groups()[0]
test_bg.build_index(k=4)
print("Index built on specific block group")

### 5.5 Clear Index

In [ ]:
repo.clear_index()
print("All search indexes cleared")

## 6. Graph Operations

Deriving subgraphs and chunks from block groups.

### 6.1 Derive Subgraph

In [ ]:
bg = repo.get_block_groups()[0]
subgraph = gen.derive_subgraph(repo, bg, 0, 100)
print(f"Derived subgraph type: {type(subgraph).__name__}")
print(f"  Number of nodes: {len(subgraph.nodes)}")
print(f"  Number of edges: {len(subgraph.edges)}")

### 6.2 Derive Chunks

In [ ]:
chunks = gen.derive_chunks(repo, bg, chunk_size=50)
print(f"Derived {len(chunks)} chunks from {bg}")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {chunk}")

## 7. Interactive Graph Visualization

The Jupyter widget allows interactive exploration of sequence graphs.

In [ ]:
bg = repo.get_block_groups()[0]
widget = bg.plot(rows=20, cols=80, detail="full")
widget

### 7.1 Highlight matches in widget

In [ ]:
matches = repo.search(bg, "ATCG")
print(f"Found {len(matches)} matches to highlight")

widget.clear_highlights()
for m in matches[:5]:
    widget.highlight_match(m, color="yellow")
print("Highlighted matches shown in widget")

### 7.2 Different highlight colors

In [ ]:
widget.clear_highlights()
widget.highlight_match(matches[0], color="cyan")
widget.highlight_match(matches[1] if len(matches) > 1 else matches[0], color="red")
print("Multi-color highlights applied")

### 7.3 Freeze the widget

`widget.freeze()` locks the widget into a static snapshot.  It captures the
current canvas as a PNG and embeds it via `_repr_html_`, so the graph remains
visible in static viewers (GitHub, nbviewer) without the `gen` module running.
All further interaction calls become no-ops.

In [ ]:
widget.freeze()
print("Widget frozen — PNG embedded in cell output.")

## 8. Transactions

Group multiple operations into atomic transactions.

In [ ]:
transaction_repo = gen.Repository(str(tempfile.mkdtemp(prefix="gen-txn-")))

with transaction_repo.transaction() as txn:
    txn.import_fasta(str(FIXTURES / "simple.fa"), name="txn1", collection="txn")
    txn.import_fasta(str(FIXTURES / "parts.fa"), name="txn2", collection="txn")
    print("Transaction committed successfully")

bgs = transaction_repo.get_block_groups()
print(f"Block groups after transaction: {len(bgs)}")

### 8.1 Transaction rollback on error

In [ ]:
rollback_repo = gen.Repository(str(tempfile.mkdtemp(prefix="gen-rollback-")))

try:
    with rollback_repo.transaction() as txn:
        txn.import_fasta(str(FIXTURES / "simple.fa"), name="should_exist", collection="test")
        txn.import_fasta(str(FIXTURES / "simple.fa"), name="should_exist", collection="test")  # duplicate
except Exception as e:
    print(f"Expected error: {e}")

bgs = rollback_repo.get_block_groups()
print(f"Block groups after rollback: {len(bgs)}")

## 9. Block Group Properties and Methods

In [ ]:
bg = repo.get_block_groups()[0]
print(f"Block Group: {bg}")
print(f"  ID: {bg.id}")
print(f"  Name: {bg.name}")
print(f"  Collection: {bg.collection_name}")
print(f"  Sample: {bg.sample_name}")

### 9.1 Get block sequence

In [ ]:
blocks = repo.query("SELECT node_id, sequence_start, sequence_end FROM blocks WHERE block_group_id = ? LIMIT 5", [str(bg.id)])
print(f"Found {len(blocks)} blocks")
for block_row in blocks:
    print(f"  Block: {block_row}")
    if block_row:
        block = gen.PyBlock(
            bg.id if len(block_row[0].split()) > 0 else bg.id,  # node_id 
            block_row[1] if len(block_row) > 1 else 0,  # start
            block_row[2] if len(block_row) > 2 else 0   # end
        )
        # This is just for demonstration - in practice you'd use the right node_id

### 9.2 Graph position and locus

In [ ]:
pos = gen.PyGraphPos(block_id=bg.id, offset=5)
print(f"Graph position: {pos}")
print(f"  Block ID: {pos.block_id}")
print(f"  Offset: {pos.offset}")

locus = gen.PyGraphLocus(positions=[pos])
print(f"Locus: {locus}")
print(f"  Positions: {locus.positions}")

## 10. Collection Operations

In [ ]:
result = repo.query("SELECT DISTINCT collection_name FROM block_groups")
collections = [row[0] for row in result]
print(f"Collections: {collections}")

for coll in collections:
    bgs = repo.get_block_groups_by_collection(coll)
    print(f"  {coll}: {len(bgs)} block group(s)")

## 11. Database Queries

In [ ]:
result = repo.query("SELECT COUNT(*) FROM block_groups")
print(f"Block groups count: {result[0][0]}")

result = repo.query("SELECT name, collection_name FROM block_groups LIMIT 5")
print("Block group names and collections:")
for row in result:
    print(f"  {row[0]} in {row[1]}")

## 12. Make Stitch (create joined sequence)

In [ ]:
bgs = repo.get_block_groups()[:3]
print(f"Creating stitch from {len(bgs)} block groups")

stitch_result = gen.make_stitch(repo, [bg.id for bg in bgs])
print(f"Stitch result: {stitch_result}")

## 13. Graph Export Formats

Convert graphs to other formats for external tools.

### 13.1 NetworkX conversion

In [ ]:
try:
    import networkx as nx
    
    bg = repo.get_block_groups()[0]
    nx_graph = repo.block_group_to_networkx(bg)
    print(f"NetworkX graph: {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")
    print(f"Is directed: {nx_graph.is_directed()}")
    
    degrees = [d for n, d in nx_graph.degree()]
    print(f"Average degree: {sum(degrees)/len(degrees):.2f}")
    
except ImportError:
    print("Install networkx: pip install networkx")

### 13.2 RustworkX conversion

In [ ]:
try:
    import rustworkx as rx
    
    bg = repo.get_block_groups()[0]
    rx_graph = repo.block_group_to_rustworkx(bg)
    print(f"RustworkX graph: {rx_graph.num_nodes()} nodes, {rx_graph.num_edges()} edges")
    
except ImportError:
    print("Install rustworkx: pip install rustworkx")

### 13.3 Dictionary representation

In [ ]:
bg = repo.get_block_groups()[0]
dict_repr = repo.block_group_to_dict(bg)
print(f"Dictionary keys: {list(dict_repr.keys())}")
print(f"Nodes: {len(dict_repr['nodes'])} items")
print(f"Edges: {len(dict_repr['edges'])} items")

## 14. Package Information

In [ ]:
print(f"Gen version: {gen.__version__}")
print(f"\nExported functions and classes:")
for name in sorted(gen.__all__):
    print(f"  - {name}")

## Summary

This notebook demonstrated:

1. **Repository initialization** - `gen.Repository()`, `gen.init()`, `gen.get_gen_dir()`
2. **Import functions** - `import_fasta`, `import_gfa`, `import_genbank`, `import_library`
3. **Export functions** - `export_fasta`, `export_gfa`, `export_genbank`
4. **Update functions** - `update_with_sequence`, `update_with_fasta`, `update_with_vcf`, `update_with_gfa`, `update_with_genbank`, `update_with_gaf`, `update_with_library`
5. **Search and indexing** - `search`, `build_index`, `clear_index`
6. **Graph operations** - `derive_subgraph`, `derive_chunks`
7. **Interactive visualization** - Jupyter widget with highlighting — including `freeze()` for static distribution
8. **Transactions** - Atomic operations with rollback on error
9. **Block group properties** - ID, name, collection_name, sample_name
10. **Graph position/types** - PyGraphPos, PyGraphLocus, PyBlock
11. **Database queries** - Direct SQL queries on the repository
12. **Graph conversions** - NetworkX, RustworkX, dictionary formats

---